# VUMPS from finite matrix contractions

An infinite chain has an extensive energy, but changing one tensor requires
only finite matrices describing the two half chains. VUMPS optimizes a site
centre and a bond centre in those environments, then updates the canonical
tensors so that the centres describe the same uniform state.

This notebook follows **Infinite tensor networks**, particularly **Variational
uniform MPS (VUMPS)** and **A product-state example**, in the
[lecture notes](https://arxiv.org/abs/2512.02706).
We first inspect a local optimization from an exactly canonical $D=2$ state.
The Ising examples then repeat the complete update to convergence at $D=1$
and $D=2$. The second calculation reconstructs a compatible uniform state
after every update.
All numerical operations use NumPy. Python 3.10 or later and NumPy 1.26 or
later are sufficient.

As in the first notebook, `A[a,s,b]` has shape $(D,d,D)$, with left bond $a$,
physical index $s$, and right bond $b$. The mathematical matrix $A^s$ is
`A[:,s,:]`. The bond matrix `C[a,b]` has shape $(D,D)$.


In [1]:
import numpy as np
np.set_printoptions(precision=8, suppress=True)
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.diag([1, -1]).astype(complex)

def ising_bond(J=1.0, hx=1.2, hz=0.0):
    matrix = -J*np.kron(Z, Z)
    matrix -= hx/2*(np.kron(X, I2) + np.kron(I2, X))
    matrix -= hz/2*(np.kron(Z, I2) + np.kron(I2, Z))
    return matrix.reshape(2, 2, 2, 2)

h = ising_bond()
print("Local interaction shape:", h.shape)


Local interaction shape: (2, 2, 2, 2)


## An exactly canonical state at $D=2$

The canonical relations are
$$\sum_s A_L^{s\dagger}A_L^s=I,\qquad
\sum_s A_R^s A_R^{s\dagger}=I,\qquad
A_C^s=A_L^sC=CA_R^s.$$
We normalize $\operatorname{Tr}(C^\dagger C)=1$. To construct a state with
known fixed points, choose
$$A_L^0=\begin{pmatrix}\cos u&0\\0&\cos v\end{pmatrix},\qquad
A_L^1=e^{i\eta}\begin{pmatrix}0&\sin v\\\sin u&0\end{pmatrix}.$$
The left-isometry condition follows by summing the squared columns. A diagonal
fixed point $\rho=\operatorname{diag}(r_0,r_1)$ obeys
$r_0\sin^2u=r_1\sin^2v$ and $r_0+r_1=1$.
Set $C=\sqrt\rho$ and solve $CA_R^s=A_L^sC$. Its two nonzero Schmidt values
make the half chains entangled.




In [2]:
u, v, eta = 0.35, 0.8, 0.23
AL = np.stack([np.diag([np.cos(u), np.cos(v)]),
               np.exp(1j*eta)*np.array([[0, np.sin(v)], [np.sin(u), 0]])], axis=1)
weights = np.array([np.sin(v)**2, np.sin(u)**2])
weights /= weights.sum()
C = np.diag(np.sqrt(weights)).astype(complex)
AR = np.stack([np.linalg.solve(C, AL[:, s, :] @ C) for s in range(2)], axis=1)
AC = np.einsum("asb,bc->asc", AL, C)
assert np.allclose(AC, np.einsum("ab,bsc->asc", C, AR))
assert np.allclose(np.einsum("asb,asc->bc", AL.conj(), AL), I2)
assert np.allclose(np.einsum("asb,csb->ac", AR, AR.conj()), I2)
assert np.isclose(np.linalg.norm(AC), 1)
print("AL, AR, AC shapes:", AL.shape, AR.shape, AC.shape)
print("Schmidt probabilities:", weights)


AL, AR, AC shapes: (2, 2, 2) (2, 2, 2) (2, 2, 2)
Schmidt probabilities: [0.81401001 0.18598999]


## Transfer maps and the nondecaying identity

Contracting one vacuum site defines
$$\mathcal T_L(Q)=\sum_s A_L^{s\dagger}QA_L^s,\qquad
\mathcal T_R(Q)=\sum_s A_R^sQ A_R^{s\dagger}.$$
Both maps preserve the identity. Their dual fixed points are
$\rho_L=CC^\dagger$ and $\rho_R=C^\dagger C$ for the compatible tensors above.
For example, $\sum_s A_L^s\rho_L A_L^{s\dagger}=\rho_L$ ensures
$\operatorname{Tr}[\rho_L\mathcal T_L(Q)]=\operatorname{Tr}(\rho_LQ)$.
The next code retains the matrix form of the contraction.




In [3]:
def transfer_left(tensor, matrix):
    return sum(tensor[:, s, :].conj().T @ matrix @ tensor[:, s, :]
               for s in range(tensor.shape[1]))

def transfer_right(tensor, matrix):
    return sum(tensor[:, s, :] @ matrix @ tensor[:, s, :].conj().T
               for s in range(tensor.shape[1]))

TL = lambda matrix: transfer_left(AL, matrix)
TR = lambda matrix: transfer_right(AR, matrix)
rhoL, rhoR = C @ C.conj().T, C.conj().T @ C
assert np.allclose(TL(I2), I2)
assert np.allclose(TR(I2), I2)
assert np.allclose(transfer_right(AL, rhoL), rhoL)
assert np.allclose(transfer_left(AR, rhoR), rhoR)


At these small dimensions we can also store a linear map as a dense matrix.
Its $j$th column is the flattened output obtained by applying the map to the
$j$th basis array. NumPy's default flattening lists rows, so
$\operatorname{vec}(LQR)=(L\otimes R^T)\operatorname{vec}(Q)$.
This supplies an independent Kronecker check of the transfer matrix.
The identity eigenvalue must be unique and the remaining eigenvalues have
modulus below one for the environment sum used below.


In [4]:
def dense_map(action, input_shape):
    size = int(np.prod(input_shape))
    basis = np.eye(size, dtype=complex)
    return np.column_stack([np.asarray(action(row.reshape(input_shape))).reshape(-1)
                            for row in basis])

transfer_matrix = dense_map(TL, (2, 2))
kronecker_matrix = sum(np.kron(AL[:, s, :].conj().T, AL[:, s, :].T)
                       for s in range(2))
assert np.allclose(transfer_matrix, kronecker_matrix)
transfer_eigenvalues = np.linalg.eigvals(transfer_matrix)
assert np.count_nonzero(np.isclose(transfer_eigenvalues, 1)) == 1
assert np.sort(abs(transfer_eigenvalues))[-2] < 1
print("Transfer eigenvalues:", np.sort(transfer_eigenvalues.real))


Transfer eigenvalues: [0.36782133 0.40848744 0.9004471  1.        ]


## A local energy and its two half-chain sources

The interaction is indexed by `h[s,t,u,v]`$=\langle st|h|uv\rangle$.
The two-site centre is $\Theta^{st}=A_L^sCA_R^t$ and has shape $(d,d,D,D)$.
The energy per site is
$$e=\sum_{stuv}h_{st,uv}\operatorname{Tr}
 [\Theta^{st\dagger}\Theta^{uv}].$$
Contracting the interaction entirely within either half chain leaves a matrix
on its inner bond:
$$b_L=\sum_{stuv}h_{st,uv}(A_L^sA_L^t)^\dagger A_L^uA_L^v,$$
$$b_R=\sum_{stuv}h_{st,uv}A_R^uA_R^v(A_R^sA_R^t)^\dagger.$$
The source energies $\operatorname{Tr}(\rho_Lb_L)$ and
$\operatorname{Tr}(\rho_Rb_R)$ must agree with the two-site calculation.


In [5]:
def bond_energy(left, centre, right, interaction):
    theta = np.einsum("asb,bc,ctd->stad", left, centre, right)
    value = np.einsum("stuv,stab,uvab->", interaction, theta.conj(), theta)
    assert abs(value.imag) < 1e-11
    return float(value.real)

def hamiltonian_sources(left, right, interaction):
    D, d, _ = left.shape
    b_left = np.zeros((D, D), dtype=complex)
    b_right = np.zeros_like(b_left)
    for s, t, u, v in np.ndindex(d, d, d, d):
        coefficient = interaction[s, t, u, v]
        b_left += coefficient*(left[:, s, :] @ left[:, t, :]).conj().T @ (left[:, u, :] @ left[:, v, :])
        b_right += coefficient*(right[:, u, :] @ right[:, v, :]) @ (right[:, s, :] @ right[:, t, :]).conj().T
    return b_left, b_right

e = bond_energy(AL, C, AR, h)
bL, bR = hamiltonian_sources(AL, AR, h)
assert np.allclose([np.trace(rhoL @ bL), np.trace(rhoR @ bR)], e)
print(f"Energy per site = {e:.10f}")


Energy per site = -0.4763410580


## Removing the extensive energy

An interaction $n$ sites from the centre contributes $\mathcal T_L^n(b_L)$.
Summing to a distance $M$ produces $MeI$ plus a finite remainder. The source
$b_0=b_L-eI$ obeys $\operatorname{Tr}(\rho_Lb_0)=0$, so it has no component
along the nondecaying identity. Its convergent sum satisfies
$(\mathrm{Id}-\mathcal T_L)K_L=b_0$.

The equation alone permits $K_L\mapsto K_L+cI$. Define
$\mathcal P_L(Q)=I\operatorname{Tr}(\rho_LQ)$ and solve
$$[\mathrm{Id}-\mathcal T_L+\mathcal P_L]K_L=b_L-eI.$$
Contracting with $\rho_L$ enforces $\operatorname{Tr}(\rho_LK_L)=0$.
The right environment is obtained in the same way. Here the dense system
has only $D^2=4$ unknowns.




In [6]:
def solve_environment(transfer, density, source):
    D = density.shape[0]
    identity = np.eye(D, dtype=complex)
    energy = np.trace(density @ source)
    assert abs(energy.imag) < 1e-11
    energy = float(energy.real)
    projected = lambda matrix: matrix-transfer(matrix)+identity*np.trace(density @ matrix)
    system = dense_map(projected, (D, D))
    result = np.linalg.solve(system, (source-energy*identity).reshape(-1)).reshape(D, D)
    assert np.allclose(projected(result), source-energy*identity)
    assert abs(np.trace(density @ result)) < 1e-10
    assert np.allclose(result, result.conj().T)
    return result, energy

KL, eL = solve_environment(TL, rhoL, bL)
KR, eR = solve_environment(TR, rhoR, bR)
assert np.allclose([eL, eR], e)
print("Left environment:\n", KL)
print("Right environment:\n", KR)


Left environment:
 [[ -0.31953794+0.j         -10.17939077-0.14167302j]
 [-10.17939077+0.14167302j   1.3985004 +0.j        ]]
Right environment:
 [[ -0.31953794+0.j         -10.17939077+0.14167302j]
 [-10.17939077-0.14167302j   1.3985004 +0.j        ]]


The projected solve can be checked without solving another linear system.
Add the terms $b_0,\mathcal T_L(b_0),\ldots$ directly. The transfer spectrum
controls the rate at which this finite sum approaches $K_L$. This check also
tests the subtraction of $eI$ and the zero-trace convention.


In [7]:
term = bL-e*I2
partial_sum = np.zeros_like(term)
for distance in range(400):
    partial_sum += term
    term = TL(term)
assert np.allclose(partial_sum, KL, atol=2e-11)
assert np.linalg.norm(KL) > 1e-3
print("Geometric-sum error:", np.linalg.norm(partial_sum-KL))
print("Weighted traces:", np.trace(rhoL @ KL), np.trace(rhoR @ KR))


Geometric-sum error: 2.2157072537628984e-14
Weighted traces: (-5.551115123125783e-17+0j) (-1.1102230246251565e-16+0j)


## The site and bond effective Hamiltonians

For a trial site tensor $Q[a,s,b]$ and bond matrix $R[a,b]$, the book gives
$$[\mathcal H_A(Q)]^s=K_LQ^s+Q^sK_R
+\sum_{tuv}h_{ts,uv}A_L^{t\dagger}A_L^uQ^v
+\sum_{tuv}h_{st,uv}Q^uA_R^vA_R^{t\dagger},$$
$$\mathcal H_C(R)=K_LR+RK_R+
\sum_{stuv}h_{st,uv}A_L^{s\dagger}A_L^uR A_R^vA_R^{t\dagger}.$$
There are two interactions touching a site centre and one crossing a bond
centre. The sources have already had their extensive energy removed.
The matrix elements `h` in these two actions remain those of the original
unshifted interaction.


In [8]:
def effective_actions(left, right, interaction, k_left, k_right):
    D, d, _ = left.shape
    def site_action(trial):
        result = np.stack([k_left @ trial[:, s, :] + trial[:, s, :] @ k_right
                           for s in range(d)], axis=1)
        for s, t, u, v in np.ndindex(d, d, d, d):
            coefficient = interaction[s, t, u, v]
            result[:, t, :] += coefficient*left[:, s, :].conj().T @ left[:, u, :] @ trial[:, v, :]
            result[:, s, :] += coefficient*trial[:, u, :] @ right[:, v, :] @ right[:, t, :].conj().T
        return result
    def bond_action(trial):
        result = k_left @ trial + trial @ k_right
        for s, t, u, v in np.ndindex(d, d, d, d):
            result += interaction[s, t, u, v]*left[:, s, :].conj().T @ left[:, u, :] @ trial @ right[:, v, :] @ right[:, t, :].conj().T
        return result
    return site_action, bond_action

HA, HC = effective_actions(AL, AR, h, KL, KR)
matrix_A = dense_map(HA, AC.shape)
matrix_C = dense_map(HC, C.shape)
assert np.allclose(matrix_A, matrix_A.conj().T)
assert np.allclose(matrix_C, matrix_C.conj().T)
print("Effective matrix dimensions:", matrix_A.shape, matrix_C.shape)


Effective matrix dimensions: (8, 8) (4, 4)


To verify the physical-index ordering independently, embed a trial centre in
a three-site tensor:
$$F_3(Q)^{tsv}=A_L^tQ^sA_R^v.$$
Act on its physical indices with $H_3=h\otimes I+I\otimes h$, then contract
with $F_3^\dagger$. This reconstructs the two touching-bond terms in
$\mathcal H_A$. A two-site embedding $F_2(R)^{st}=A_L^sRA_R^t$ reconstructs
the crossing term in $\mathcal H_C$. The canonical relations imply
$F_3^\dagger F_3=I$ and $F_2^\dagger F_2=I$.
The diagram displays the uncontracted ket part of $F_3$.




In [9]:
F3 = dense_map(lambda q: np.einsum("atb,bsc,cvd->tsvad", AL, q, AR), AC.shape)
F2 = dense_map(lambda r: np.einsum("asb,bc,ctd->stad", AL, r, AR), C.shape)
h_matrix = h.reshape(4, 4)
H3 = np.kron(h_matrix, I2) + np.kron(I2, h_matrix)
environment_A = np.kron(KL, np.eye(4)) + np.kron(np.eye(4), KR.T)
environment_C = np.kron(KL, I2) + np.kron(I2, KR.T)
reference_A = F3.conj().T @ np.kron(H3, np.eye(4)) @ F3 + environment_A
reference_C = F2.conj().T @ np.kron(h_matrix, np.eye(4)) @ F2 + environment_C
assert np.allclose(F3.conj().T @ F3, np.eye(8))
assert np.allclose(F2.conj().T @ F2, np.eye(4))
assert np.allclose(matrix_A, reference_A)
assert np.allclose(matrix_C, reference_C)
print("Independent site-action error:", np.linalg.norm(matrix_A-reference_A))
print("Independent bond-action error:", np.linalg.norm(matrix_C-reference_C))


Independent site-action error: 9.363683674018207e-16
Independent bond-action error: 1.7919223819685796e-15


## The two lowest-eigenvector solves

With the environments fixed, minimize the normalized quadratic form of each
effective Hamiltonian. At these dimensions `eigh` returns the full spectrum,
and its lowest eigenvector gives $\widetilde A_C$ or $\widetilde C$.
Each has Frobenius norm one. Fixing the phase of its largest component only
chooses a representative of the eigenvector. The site and bond vectors are
optimized independently, so their shapes alone do not guarantee consistency.


In [10]:
def lowest_tensor(action, shape):
    matrix = dense_map(action, shape)
    assert np.allclose(matrix, matrix.conj().T)
    energies, vectors = np.linalg.eigh(matrix)
    vector = vectors[:, 0].astype(complex)
    vector *= np.exp(-1j*np.angle(vector[np.argmax(abs(vector))]))
    result = vector.reshape(shape)
    residual = np.linalg.norm(action(result)-energies[0]*result)
    assert residual < 1e-10
    return float(energies[0]), result, residual

epsilon_A, AC_tilde, residual_A = lowest_tensor(HA, AC.shape)
epsilon_C, C_tilde, residual_C = lowest_tensor(HC, C.shape)
print("Lowest effective eigenvalues:", epsilon_A, epsilon_C)
print("Eigenvector residuals:", residual_A, residual_C)


Lowest effective eigenvalues: -21.873901362801412 -20.581394299793885
Eigenvector residuals: 8.002735794931746e-15 1.3414193543264692e-14


## Paired polar reconstruction

For the left update, reshape $\widetilde A_C$ to
$M_L[(a,s),b]$ of shape $(Dd,D)$. Right polar decompositions give
$M_L=U_LP_A$ and $\widetilde C=U_CP_C$. Then
$$A_L^{\rm new}=U_LU_C^\dagger,\qquad
\epsilon_L=\|M_L-A_L^{\rm new}\widetilde C\|_F=\|P_A-P_C\|_F.$$
If $M=W\Sigma V^\dagger$, its polar factors are
$U=WV^\dagger$ and $P=V\Sigma V^\dagger$.
This uses no inverse of $C$ and hence no division by small Schmidt values.




In [11]:
def right_polar(matrix):
    w, singular, vh = np.linalg.svd(matrix, full_matrices=False)
    unitary = w @ vh
    positive = (vh.conj().T*singular) @ vh
    return unitary, positive

def left_polar(matrix):
    w, singular, vh = np.linalg.svd(matrix, full_matrices=False)
    positive = (w*singular) @ w.conj().T
    unitary = w @ vh
    return positive, unitary


For the right update, reshape the same centre to $M_R[a,(s,b)]$ of shape
$(D,dD)$. Left polar decompositions give $M_R=Q_AV_R$ and
$\widetilde C=Q_CV_C$. The right-canonical tensor is
$A_R^{\rm new}=V_C^\dagger V_R$, and
$\epsilon_R=\|M_R-\widetilde C A_R^{\rm new}\|_F=\|Q_A-Q_C\|_F$.
The code checks both identities and both canonical isometries.


In [12]:
def paired_polar(centre, bond):
    D, d, _ = centre.shape
    ML, MR = centre.reshape(D*d, D), centre.reshape(D, d*D)
    UL, PA = right_polar(ML)
    UC, PC = right_polar(bond)
    QA, VR = left_polar(MR)
    QC, VC = left_polar(bond)
    left = (UL @ UC.conj().T).reshape(D, d, D)
    right = (VC.conj().T @ VR).reshape(D, d, D)
    error_left = np.linalg.norm(ML-left.reshape(D*d, D) @ bond)
    error_right = np.linalg.norm(MR-bond @ right.reshape(D, d*D))
    assert np.isclose(error_left, np.linalg.norm(PA-PC), atol=1e-12)
    assert np.isclose(error_right, np.linalg.norm(QA-QC), atol=1e-12)
    assert np.allclose(left.reshape(D*d, D).conj().T @ left.reshape(D*d, D), np.eye(D))
    assert np.allclose(right.reshape(D, d*D) @ right.reshape(D, d*D).conj().T, np.eye(D))
    return left, right, error_left, error_right

recovered_L, recovered_R, exact_L, exact_R = paired_polar(AC, C)
assert np.allclose(recovered_L, AL)
assert np.allclose(recovered_R, AR)
new_AL, new_AR, error_L, error_R = paired_polar(AC_tilde, C_tilde)
print("Mismatches for the original compatible state:", exact_L, exact_R)
print("Mismatches after independent optimization:", error_L, error_R)


Mismatches for the original compatible state: 1.2427091343074311e-16 1.2412670766236366e-16
Mismatches after independent optimization: 0.010575205004160045 0.010575205004159973


The updated left and right tensors are isometries. Their nonzero centre
mismatches measure the remaining incompatibility of the two optimized
centres. The following energy-residual identity is evaluated on the original
compatible $D=2$ state, whose environments solve the stated equations.

Contracting the site action with $A_L^\dagger$ gives
$$\sum_s A_L^{s\dagger}[\mathcal H_A(A_C)]^s=\mathcal H_C(C)+eC.$$
Thus, with $\mathbb A_L[(a,s),b]=(A_L^s)_{ab}$,
$$G^s=[\mathcal H_A(A_C)]^s-A_L^s\mathcal H_C(C)-eA_C^s$$
is exactly $(I-\mathbb A_L\mathbb A_L^\dagger)\mathcal H_A(A_C)$.
Its Frobenius norm is the uniform projected energy residual. It need not
vanish for our constructed state.


In [13]:
def gradient_tensor(left, centre, bond, energy, site_action, bond_action):
    return (site_action(centre)
            - np.einsum("asb,bc->asc", left, bond_action(bond))-energy*centre)

site_output = HA(AC)
contraction = np.einsum("asb,asc->bc", AL.conj(), site_output)
assert np.allclose(contraction, HC(C)+e*C)
G = gradient_tensor(AL, AC, C, e, HA, HC)
stacked = AL.reshape(4, 2)
projected = (np.eye(4)-stacked @ stacked.conj().T) @ site_output.reshape(4, 2)
assert np.allclose(G.reshape(4, 2), projected)
assert np.allclose(np.einsum("asb,asc->bc", AL.conj(), G), 0)
assert np.isclose(np.vdot(AC, HA(AC)), 2*e)
assert np.isclose(np.vdot(C, HC(C)), e)
print("Projected gradient norm:", np.linalg.norm(G))


Projected gradient norm: 4.535917336479126


## A complete convergent $D=1$ calculation

For a product state $|a\rangle^{\otimes\infty}$, take
$A_L^s=A_R^s=A_C^s=a_s$ and $C=1$. The energy is
$$e=-Jm_z^2-h_xm_x-h_zm_z,\qquad
m_x=\langle a|X|a\rangle,\quad m_z=\langle a|Z|a\rangle.$$
The transfer maps act on scalars and $b_L=b_R=e$, so $K_L=K_R=0$.
The centre still feels the two neighbouring spins through
$$\mathcal H_A=-h_xX-(2Jm_z+h_z)Z-(h_xm_x+h_zm_z)I.$$
We first compare this analytic $2\times2$ matrix with the generic contractions.


In [14]:
def product_problem(a, interaction):
    a = np.asarray(a, dtype=complex)/np.linalg.norm(a)
    tensor = a.reshape(1, len(a), 1)
    bond = np.ones((1, 1), dtype=complex)
    density = bond.copy()
    b_left, b_right = hamiltonian_sources(tensor, tensor, interaction)
    k_left, energy = solve_environment(lambda q: transfer_left(tensor, q), density, b_left)
    k_right, other_energy = solve_environment(lambda q: transfer_right(tensor, q), density, b_right)
    assert np.isclose(energy, other_energy)
    site_action, bond_action = effective_actions(tensor, tensor, interaction, k_left, k_right)
    return tensor, bond, energy, site_action, bond_action

initial = np.sqrt([(1+0.5)/2, (1-0.5)/2])
product_A, product_C, product_e, product_HA, product_HC = product_problem(initial, h)
mx, mz = np.vdot(initial, X @ initial).real, np.vdot(initial, Z @ initial).real
analytic_A = -1.2*X-2*mz*Z-1.2*mx*I2
assert np.allclose(dense_map(product_HA, product_A.shape), analytic_A)
assert np.allclose(product_HC(product_C), product_e*product_C)
print("Initial mx, mz, energy:", mx, mz, product_e)


Initial mx, mz, energy: 0.8660254037844386 0.4999999999999999 -1.2892304845413276


One update solves both eigenproblems and applies both polar reconstructions.
At $D=1$, unit norms make the two polar mismatches vanish. The new site
amplitudes therefore already define a compatible product state.
Rebuild its environments and repeat until $\|G\|_F$ is small.
The energy difference alone would become small earlier because the energy
changes quadratically near a stationary point.


In [15]:
def solve_product(J=1.0, hx=1.2, hz=0.0, initial_mz=0.5, tolerance=1e-11, maxiter=1000):
    interaction = ising_bond(J, hx, hz)
    a = np.sqrt([(1+initial_mz)/2, (1-initial_mz)/2]).astype(complex)
    history = []
    for iteration in range(maxiter):
        tensor, bond, energy, site_action, bond_action = product_problem(a, interaction)
        residual = np.linalg.norm(gradient_tensor(tensor, tensor, bond, energy, site_action, bond_action))
        mx, mz = np.vdot(a, X @ a).real, np.vdot(a, Z @ a).real
        history.append((iteration, energy, mx, mz, residual))
        if residual < tolerance:
            return a, interaction, np.array(history)
        _, centre_tilde, _ = lowest_tensor(site_action, tensor.shape)
        _, bond_tilde, _ = lowest_tensor(bond_action, bond.shape)
        left, right, mismatch_L, mismatch_R = paired_polar(centre_tilde, bond_tilde)
        assert max(mismatch_L, mismatch_R) < 1e-12
        assert np.allclose(left, right)
        a = left[0, :, 0]
    raise RuntimeError("Product-state iteration did not converge")

a_final, h_final, history = solve_product()
print("iteration        energy             mx          mz          ||G||")
rows = list(range(min(5, len(history)))) + [len(history)-1]
for row in rows:
    n, energy, mx, mz, residual = history[row]
    print(f"{int(n):6d}    {energy: .12f}    {mx:.8f}  {mz:.8f}  {residual:.3e}")


iteration        energy             mx          mz          ||G||
     0    -1.289230484541    0.86602540  0.50000000  2.660e-01
     1    -1.331701601091    0.76822128  0.64018440  2.154e-01
     2    -1.352971481906    0.68383626  0.72963551  1.223e-01
     3    -1.358764173106    0.63515433  0.77238525  5.431e-02
     4    -1.359818654049    0.61346648  0.78972076  2.127e-02
    26    -1.360000000000    0.60000000  0.80000000  3.874e-12


The minimum within the product-state family can be found independently.
For $h_z=0$, $J>0$, and positive amplitudes, $m_x^2+m_z^2=1$ and
$$e=-J+Jm_x^2-h_xm_x
=-J-\frac{h_x^2}{4J}+J\left(m_x-\frac{h_x}{2J}\right)^2.$$
For $0<h_x<2J$, the positive-magnetization minimum is
$m_x=h_x/(2J)$ and $m_z=\sqrt{1-m_x^2}$.
At $J=1,h_x=1.2$, this gives $(m_x,m_z)=(0.6,0.8)$ and $e=-1.36$.
These are the exact product variational values. Entanglement at larger $D$
enlarges the variational space.

In the unshifted-interaction convention, stationarity also requires
$\varepsilon_A=2e$ and $\varepsilon_C=e$.


In [16]:
final_A, final_C, final_e, final_HA, final_HC = product_problem(a_final, h_final)
final_eps_A, _, _ = lowest_tensor(final_HA, final_A.shape)
final_eps_C, _, _ = lowest_tensor(final_HC, final_C.shape)
assert np.allclose(history[-1, 2:4], [0.6, 0.8], atol=1e-10)
assert np.isclose(final_e, -1.36, atol=1e-12)
assert np.allclose([final_eps_A, final_eps_C], [2*final_e, final_e], atol=1e-11)
print("Converged product energy:", final_e)
print("Effective eigenvalues:", final_eps_A, final_eps_C)


Converged product energy: -1.3599999999999997
Effective eigenvalues: -2.7199999999999998 -1.3599999999999997


## Reconstructing a consistent entangled state after an update

The left-canonical tensor returned by the paired polar step can itself be
repeated to define a uniform state. We can obtain its compatible bond and
right-canonical tensors by contracting its transfer map to convergence.
This gives a small-$D$ VUMPS iteration with explicit canonical reconstruction
after each local update.

Let $\mathcal E_L(Q)=\sum_s A_L^sQA_L^{s\dagger}$. The left-isometry
condition makes this map trace preserving. Its trace-one fixed point $r$
can be found from
$$[\mathrm{Id}-\mathcal E_L+\mathcal P_{\rm tr}]r=I/D,
\qquad \mathcal P_{\rm tr}(Q)=I\operatorname{Tr}(Q)/D.$$
Taking the trace gives $\operatorname{Tr}(r)=1$. Substitution then gives
$\mathcal E_L(r)=r$. For an injective full-rank state, factor
$r=CC^\dagger$ by taking its positive square root. Solve
$CA_R^s=A_L^sC$ for each physical index. The resulting tensor is right
canonical because
$$\sum_s A_R^sA_R^{s\dagger}
=C^{-1}\left(\sum_s A_L^srA_L^{s\dagger}\right)(C^{-1})^\dagger
=C^{-1}r(C^{-1})^\dagger=I.$$
The matrix solve below is used only on this full-rank example. The retained
fixed-point eigenvalues are checked explicitly.


In [17]:
def canonical_from_left(left):
    D, d, _ = left.shape
    identity = np.eye(D, dtype=complex)
    fixed_point_equation = lambda q: (q-transfer_right(left, q)
                                      + np.trace(q)*identity/D)
    system = dense_map(fixed_point_equation, (D, D))
    density = np.linalg.solve(system, (identity/D).reshape(-1)).reshape(D, D)
    assert np.allclose(density, density.conj().T)
    values, vectors = np.linalg.eigh((density+density.conj().T)/2)
    assert values.min() > 1e-12
    bond = (vectors*np.sqrt(values)) @ vectors.conj().T
    right = np.stack([np.linalg.solve(bond, left[:, s, :] @ bond)
                      for s in range(d)], axis=1)
    centre = np.einsum("asb,bc->asc", left, bond)
    assert np.allclose(centre, np.einsum("ab,bsc->asc", bond, right))
    assert np.allclose(np.einsum("asb,csb->ac", right, right.conj()), identity)
    assert np.isclose(np.linalg.norm(bond), 1)
    return right, bond, centre

right_check, bond_check, centre_check = canonical_from_left(AL)
assert np.allclose(bond_check, C)
assert np.allclose(right_check, AR)
print("Reconstructed Schmidt probabilities:", np.linalg.eigvalsh(bond_check @ bond_check.conj().T))


Reconstructed Schmidt probabilities: [0.18598999 0.81401001]


Each iteration now starts with one compatible state. Rebuild its sources,
environments and effective actions, solve the two eigenproblems, and compute
the paired polar update. The updated left-canonical tensor determines the
next uniform state. Its right tensor and bond matrix are reconstructed by
the fixed-point solve above. Both polar mismatches still measure whether the
independently optimized centres have reached consistency.

The convergence checks include $G$, both mismatches, and the residuals of the
**current compatible centres**,
$\|\mathcal H_A(A_C)-2eA_C\|_F$ and
$\|\mathcal H_C(C)-eC\|_F$. These last residuals are distinct from the tiny
inner eigensolver errors for $\widetilde A_C,\widetilde C$. Divide the
energy-valued residuals by $\|h\|$ before comparing them with the
dimensionless mismatches.


In [18]:
def uniform_problem(left, interaction):
    right, bond, centre = canonical_from_left(left)
    density_L, density_R = bond @ bond.conj().T, bond.conj().T @ bond
    source_L, source_R = hamiltonian_sources(left, right, interaction)
    env_L, energy = solve_environment(lambda q: transfer_left(left, q), density_L, source_L)
    env_R, energy_R = solve_environment(lambda q: transfer_right(right, q), density_R, source_R)
    assert np.isclose(energy, energy_R)
    assert np.isclose(energy, bond_energy(left, bond, right, interaction))
    site_action, bond_action = effective_actions(left, right, interaction, env_L, env_R)
    return right, bond, centre, energy, site_action, bond_action

def solve_uniform(initial_left, interaction, tolerance=1e-11, maxiter=300):
    left = initial_left.copy()
    d = interaction.shape[0]
    energy_scale = np.linalg.norm(interaction.reshape(d*d, d*d), 2)
    history = []
    for iteration in range(maxiter):
        right, bond, centre, energy, site_action, bond_action = uniform_problem(left, interaction)
        gradient = np.linalg.norm(gradient_tensor(left, centre, bond, energy, site_action, bond_action))
        residual_A = np.linalg.norm(site_action(centre)-2*energy*centre)
        residual_C = np.linalg.norm(bond_action(bond)-energy*bond)
        _, optimized_centre, _ = lowest_tensor(site_action, centre.shape)
        _, optimized_bond, _ = lowest_tensor(bond_action, bond.shape)
        updated_left, _, mismatch_L, mismatch_R = paired_polar(optimized_centre, optimized_bond)
        history.append((iteration, energy, gradient, mismatch_L, mismatch_R, residual_A, residual_C))
        error = max(gradient/energy_scale, residual_A/energy_scale, residual_C/energy_scale,
                    mismatch_L, mismatch_R)
        if error < tolerance:
            return (left, right, bond, centre), np.array(history)
        left = updated_left
    raise RuntimeError("Uniform iteration did not converge")

entangled_state, entangled_history = solve_uniform(AL, h)
print("iteration        energy          ||G||       mismatch L     centre A residual")
for row in [0, 1, 2, 5, 10, 20, len(entangled_history)-1]:
    n, energy, gradient, mismatch_L, mismatch_R, residual_A, residual_C = entangled_history[row]
    print(f"{int(n):6d}    {energy:.12f}   {gradient:.3e}   {mismatch_L:.3e}       {residual_A:.3e}")


iteration        energy          ||G||       mismatch L     centre A residual
     0    -0.476341058028   4.536e+00   1.058e-02       2.076e+01
     1    -1.324023459376   6.660e-01   1.802e-02       9.677e-01
     2    -1.405821138183   7.760e-02   1.494e-02       1.467e-01
     5    -1.417333623347   2.204e-02   5.845e-03       5.331e-02
    10    -1.418825085715   2.271e-03   6.313e-04       6.077e-03
    20    -1.418842473992   2.120e-05   5.929e-06       5.713e-05
    53    -1.418842475525   4.306e-12   1.205e-12       1.160e-11


The converged state has two nonzero Schmidt probabilities. Its energy can be
compared with the exact product variational minimum already obtained above.
We also measure the one-site magnetizations by contracting
$\rho_{st}=\sum_{a,b}(A_C)_{asb}\overline{(A_C)_{atb}}$.
Every quantity is evaluated after reconstructing the compatible centre.


In [19]:
ground_L, ground_R, ground_C, ground_AC = entangled_state
entangled_energy = bond_energy(ground_L, ground_C, ground_R, h)
schmidt_probabilities = np.linalg.svd(ground_C, compute_uv=False)**2
one_site_density = np.einsum("asb,atb->st", ground_AC, ground_AC.conj())
entanglement = -np.sum(schmidt_probabilities*np.log(schmidt_probabilities))
assert entangled_energy < final_e
assert schmidt_probabilities.min() > 0.01
assert np.allclose(ground_AC, np.einsum("ab,bsc->asc", ground_C, ground_R))
assert max(entangled_history[-1, 2:]) < 5e-11
print(f"D=1 product energy: {final_e:.12f}")
print(f"D=2 energy:         {entangled_energy:.12f}")
print("D=2 Schmidt probabilities:", schmidt_probabilities)
print(f"Entanglement entropy: {entanglement:.9f} nats")
print("mx, mz:", np.trace(one_site_density @ X).real, np.trace(one_site_density @ Z).real)


D=1 product energy: -1.360000000000
D=2 energy:         -1.418842475525
D=2 Schmidt probabilities: [0.93627768 0.06372232]
Entanglement entropy: 0.237089083 nats
mx, mz: 0.797033588245264 5.57766055564457e-11


An independent stationarity check perturbs the uniform tensor itself.
For a small complex direction $B$, form $M=(A_L+\delta B)_{(a,s),b}$ and
replace it by its isometric polar factor. At $\delta=0$ this factor equals
$A_L$, so the curve passes through the converged state. Reconstruct its
fixed point and calculate the two-site energy directly, without using either
effective Hamiltonian. The symmetric difference
$[e(\delta)-e(-\delta)]/(2\delta)$ should vanish along each sampled direction.
The polar factor preserves the isometry without imposing an arbitrary QR
column phase on this perturbation.


In [20]:
check_rng = np.random.default_rng(173)
delta = 2e-5
derivatives = []
for sample in range(4):
    direction = check_rng.normal(size=ground_L.shape)+1j*check_rng.normal(size=ground_L.shape)
    direction /= np.linalg.norm(direction)
    energies = []
    for sign in [-1, 1]:
        trial_matrix = (ground_L+sign*delta*direction).reshape(4, 2)
        trial_L = right_polar(trial_matrix)[0].reshape(2, 2, 2)
        trial_R, trial_C, _ = canonical_from_left(trial_L)
        energies.append(bond_energy(trial_L, trial_C, trial_R, h))
    derivatives.append((energies[1]-energies[0])/(2*delta))
    assert min(energies) >= entangled_energy-1e-12
assert max(abs(np.array(derivatives))) < 1e-7
print("Independent directional energy derivatives:", " ".join(f"{value:.3e}" for value in derivatives))


Independent directional energy derivatives: -2.720e-10 1.055e-10 -4.441e-11 2.720e-10


## Exercise 1: crossing the product-state transition

Repeat the converged calculation at $h_x=0.6,1.8,2.4$ with $J=1,h_z=0$.
For $h_x\ge2J$, the constrained minimum reaches $m_x=1,m_z=0$, with
$e=-h_x$. Compare energies and magnetizations with both branches of this
piecewise product-state result. The location $h_x=2J$ belongs to the
product-state approximation.


In [21]:
for field in [0.6, 1.8, 2.4]:
    amplitudes, interaction, trace = solve_product(hx=field)
    exact_x = min(field/2, 1.0)
    exact_z = np.sqrt(1-exact_x**2)
    exact_energy = -1-field**2/4 if field < 2 else -field
    assert np.allclose(trace[-1, 2:4], [exact_x, exact_z], atol=1e-9)
    assert np.isclose(trace[-1, 1], exact_energy, atol=1e-11)
    print(f"hx={field:.1f}: e={trace[-1,1]:.10f}, mx={trace[-1,2]:.8f}, mz={trace[-1,3]:.8f}")


hx=0.6: e=-1.0900000000, mx=0.30000000, mz=0.95393920
hx=1.8: e=-1.8100000000, mx=0.90000000, mz=0.43588989
hx=2.4: e=-2.4000000000, mx=1.00000000, mz=0.00000000


## Exercise 2: shifting the local energy zero

On the original $D=2$ state, replace $h$ by $h-eI$.
The source $b_L$ shifts by $-eI$ and its energy shifts by $-e$, leaving the
projected source and both environments unchanged. The effective site matrix
shifts by $-2eI$, while the bond matrix shifts by $-eI$.
Verify these factors and check that the projected gradient is unchanged.


In [22]:
shifted_h = h-e*np.eye(4).reshape(2, 2, 2, 2)
shifted_bL, shifted_bR = hamiltonian_sources(AL, AR, shifted_h)
shifted_KL, shifted_eL = solve_environment(TL, rhoL, shifted_bL)
shifted_KR, shifted_eR = solve_environment(TR, rhoR, shifted_bR)
shifted_HA, shifted_HC = effective_actions(AL, AR, shifted_h, shifted_KL, shifted_KR)
assert np.allclose(shifted_KL, KL)
assert np.allclose(shifted_KR, KR)
assert np.allclose([shifted_eL, shifted_eR], 0)
assert np.allclose(dense_map(shifted_HA, AC.shape), matrix_A-2*e*np.eye(8))
assert np.allclose(dense_map(shifted_HC, C.shape), matrix_C-e*np.eye(4))
shifted_G = gradient_tensor(AL, AC, C, 0.0, shifted_HA, shifted_HC)
assert np.allclose(shifted_G, G)
print("Gradient change under energy shift:", np.linalg.norm(shifted_G-G))


Gradient change under energy shift: 2.7697234472106437e-16


## Exercise 3: independent complex gauges on the two half chains

Choose unitary matrices $U,V$ and transform
$A_L^s\mapsto U^\dagger A_L^sU$,
$A_R^s\mapsto V^\dagger A_R^sV$, and $C\mapsto U^\dagger CV$.
The centre becomes $U^\dagger A_C^sV$. Its singular values and physical energy
are unchanged. Recompute the environments and verify
$K_L\mapsto U^\dagger K_LU$, $K_R\mapsto V^\dagger K_RV$, and
$G^s\mapsto U^\dagger G^sV$. This check exercises the complex conjugations
with a nondiagonal bond matrix.


In [23]:
Ug = np.cos(0.37)*I2+1j*np.sin(0.37)*X
Vg = np.cos(0.29)*I2+1j*np.sin(0.29)*Z
ALg = np.stack([Ug.conj().T @ AL[:, s, :] @ Ug for s in range(2)], axis=1)
ARg = np.stack([Vg.conj().T @ AR[:, s, :] @ Vg for s in range(2)], axis=1)
Cg = Ug.conj().T @ C @ Vg
ACg = np.einsum("asb,bc->asc", ALg, Cg)
bLg, bRg = hamiltonian_sources(ALg, ARg, h)
KLg, egL = solve_environment(lambda q: transfer_left(ALg, q), Cg @ Cg.conj().T, bLg)
KRg, egR = solve_environment(lambda q: transfer_right(ARg, q), Cg.conj().T @ Cg, bRg)
HAg, HCg = effective_actions(ALg, ARg, h, KLg, KRg)
Gg = gradient_tensor(ALg, ACg, Cg, egL, HAg, HCg)
expected_Gg = np.stack([Ug.conj().T @ G[:, s, :] @ Vg for s in range(2)], axis=1)
assert np.allclose(KLg, Ug.conj().T @ KL @ Ug)
assert np.allclose(KRg, Vg.conj().T @ KR @ Vg)
assert np.allclose([egL, egR, bond_energy(ALg, Cg, ARg, h)], e)
assert np.allclose(Gg, expected_Gg)
assert np.allclose(np.linalg.svd(Cg, compute_uv=False), np.linalg.svd(C, compute_uv=False))
print("Energy after gauge change:", egL)
print("Gradient covariance error:", np.linalg.norm(Gg-expected_Gg))


Energy after gauge change: -0.4763410580282759
Gradient covariance error: 7.947030094905937e-15
